# Data Exploration - North Statistics

**Upstream:** `04_data_treatment.ipynb`. **Required input:** `data/processed/north_region_mama_2025_data_en.parquet`. **Output:** in-notebook descriptive statistics only; no pipeline artifact is generated. **Downstream:** none. Notebooks 06 and 07 independently read the same canonical output from notebook 04.


The treated North region dataset (post-merge, breast cancer filter applied, 
feature-engineered) is loaded from its canonical path under `data/processed/`. 
The repository root is resolved for execution from either the repository root 
or `notebooks/`, without requiring manual path edits.


In [1]:
from pathlib import Path
import pandas as pd

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
data_path = REPO_ROOT / "data" / "processed" / "north_region_mama_2025_data_en.parquet"

if not data_path.is_file():
    raise FileNotFoundError(
        "Missing treated dataset. Run 04_data_treatment.ipynb first. "
        f"Expected path: {data_path}"
    )

df_norte = pd.read_parquet(data_path)
print(df_norte.shape)
df_norte.head()

(310340, 119)


,ID_EVENTO_ATENCAO_SAUDE,UF_PRESTADOR,DT_REALIZACAO,CD_PROCEDIMENTO,CD_TABELA_REFERENCIA,QT_ITEM_EVENTO_INFORMADO,VL_ITEM_EVENTO_INFORMADO,VL_ITEM_PAGO_FORNECEDOR,ID_PLANO,FAIXA_ETARIA,...,category_procedure,category_procedure_group,FAIXA_ETARIA_ORDINAL,PORTE_ORDINAL,IND_SIZE_NO_BENEFICIARIES,VL_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,VL_ITEM_PAGO_FORNECEDOR_CAPPED_LOG_NORM,VL_TOTAL_REMUNERACAO_CAPPED_LOG_NORM,QT_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,IND_HIGH_SPECIFICITY_PROXY
0,6501062280,AC,2025-01-01,40901114,22,1.0,57.36,0.0,170538,20 a 29 anos,...,1,0,5,1,0,0.388590,0.0,0.430246,0.0,0
1,6501061131,AC,2025-01-01,40901114,22,1.0,53.23,0.0,192463,50 a 59 anos,...,1,0,8,1,0,0.381577,0.0,0.519055,0.0,0
2,6525247680,AC,2025-01-01,40808041,22,1.0,179.07,0.0,235079,50 a 59 anos,...,1,0,8,2,0,0.496254,0.0,0.476893,0.0,0
3,6525247689,AC,2025-01-01,40808041,22,1.0,179.07,0.0,235259,60 a 69 anos,...,1,0,9,2,0,0.496254,0.0,0.476893,0.0,0
4,6525247698,AC,2025-01-01,40808041,22,1.0,179.07,0.0,234570,50 a 59 anos,...,1,0,8,2,0,0.496254,0.0,0.476893,0.0,0


310,340 rows and 119 columns, representing all outpatient 
breast-cancer-related events across the 7 states of the North region for 2025.

### Column Classification

Every column in the treated dataset was classified as **key/identifier**, 
**temporal**, **numerical**, or **categorical**, based on semantic meaning, 
not just the pandas dtype. This distinction matters because several columns 
are stored as integers (`int64`) but represent codes or binary flags rather 
than true quantities (e.g. procedure codes, one-hot encoded indicators), so a 
dtype-only classification would misclassify them as numerical.

**Classification logic:**
- Columns manually flagged as key or temporal are labeled directly.
- Columns ending in `_ORDINAL` are treated as encoded ordinal categories.
- Integer columns restricted to `{0, 1}` are treated as binary/one-hot categorical flags.
- Remaining integer/float columns are treated as numerical.
- String/category dtype columns are treated as categorical.
- Any unhandled dtype is flagged `REVIEW MANUALLY` for explicit follow-up.

In [2]:
KEY_COLUMNS = {"ID_EVENTO_ATENCAO_SAUDE"}
TEMPORAL_COLUMNS = {"DT_REALIZACAO", "MES"}

ORDINAL_ENCODED_SUFFIX = "_ORDINAL"

classification_rows = []

for col in df_norte.columns:
    dtype = df_norte[col].dtype
    dtype_str = str(dtype)

    if col in KEY_COLUMNS:
        final_class = "key / identifier"
        reason = "unique event identifier, not an analysis variable"

    elif col in TEMPORAL_COLUMNS:
        final_class = "temporal"
        reason = "datetime dtype"

    elif col.endswith(ORDINAL_ENCODED_SUFFIX):
        final_class = "categorical (ordinal, encoded)"
        reason = "integer-encoded ordinal category"

    elif pd.api.types.is_integer_dtype(dtype):
        unique_vals = set(df_norte[col].dropna().unique())
        if unique_vals.issubset({0, 1}):
            final_class = "categorical (binary / one-hot)"
            reason = "int64 restricted to {0,1} -- one-hot encoded flag"
        else:
            final_class = "numerical"
            reason = "int64 with values beyond {0,1} -- needs manual check"

    elif pd.api.types.is_float_dtype(dtype):
        final_class = "numerical"
        reason = "float dtype"

    elif isinstance(dtype, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dtype) or dtype_str == "str":
        final_class = "categorical"
        reason = "category/object/str dtype"

    else:
        final_class = "REVIEW MANUALLY"
        reason = f"unhandled dtype: {dtype_str}"

    classification_rows.append({
        "column": col,
        "pandas_dtype": dtype_str,
        "n_unique": df_norte[col].nunique(),
        "final_classification": final_class,
        "reason": reason,
    })

df_classification = pd.DataFrame(classification_rows)
print(f"Total columns classified: {len(df_classification)}")
df_classification

Total columns classified: 119


,column,pandas_dtype,n_unique,final_classification,reason
0,ID_EVENTO_ATENCAO_SAUDE,str,291864,key / identifier,"unique event identifier, not an analysis variable"
1,UF_PRESTADOR,category,7,categorical,category/object/str dtype
2,DT_REALIZACAO,datetime64[us],12,temporal,datetime dtype
3,CD_PROCEDIMENTO,category,155,categorical,category/object/str dtype
4,CD_TABELA_REFERENCIA,category,3,categorical,category/object/str dtype
...,...,...,...,...,...
114,VL_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,float64,5887,numerical,float dtype
115,VL_ITEM_PAGO_FORNECEDOR_CAPPED_LOG_NORM,float64,268,numerical,float dtype
116,VL_TOTAL_REMUNERACAO_CAPPED_LOG_NORM,float64,26497,numerical,float dtype
117,QT_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,float64,207,numerical,float dtype


**Findings:** all 119 retained columns were successfully classified with no 
column left in `REVIEW MANUALLY`: 1 key column, 2 temporal columns, 
13 numerical columns, and 103 categorical columns. Several columns that appear 
numerical by dtype (e.g. `category_drug`, `category_procedure`, 
`IND_SIZE_NO_BENEFICIARIES`) were confirmed as binary categorical flags after 
checking that their values are restricted to `{0, 1}`.

With the classification established, columns are split into type-specific 
lists to drive the appropriate descriptive statistics for each group.

In [3]:
key_cols = df_classification[df_classification["final_classification"] == "key / identifier"]["column"].tolist()
temporal_cols = df_classification[df_classification["final_classification"] == "temporal"]["column"].tolist()
numerical_cols = df_classification[df_classification["final_classification"] == "numerical"]["column"].tolist()
categorical_cols = df_classification[df_classification["final_classification"].str.startswith("categorical")]["column"].tolist()

print(f"Key columns: {len(key_cols)}")
print(f"Temporal columns: {len(temporal_cols)}")
print(f"Numerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nTotal check: {len(key_cols) + len(temporal_cols) + len(numerical_cols) + len(categorical_cols)} vs {df_classification.shape[0]} total")

Key columns: 1
Temporal columns: 2
Numerical columns: 13
Categorical columns: 103

Total check: 119 vs 119 total



Next is the descriptive statistics: mean, median, standard deviation, min/max, quartiles, 
missing rate, and skewness were computed for all 13 numerical columns.

In [4]:
numerical_stats = df_norte[numerical_cols].describe().T

# adds some metrics
numerical_stats["missing"] = df_norte[numerical_cols].isna().sum()
numerical_stats["missing_pct"] = (df_norte[numerical_cols].isna().mean() * 100).round(2)
numerical_stats["skew"] = df_norte[numerical_cols].skew()

numerical_stats = numerical_stats.round(2)
numerical_stats

,count,mean,std,min,25%,50%,75%,max,missing,missing_pct,skew
QT_ITEM_EVENTO_INFORMADO,295563.0,12.33,88.80,1.0,1.00,1.00,1.00,6000.00,14777,4.76,20.36
VL_ITEM_EVENTO_INFORMADO,295563.0,843.89,5221.69,0.0,65.66,85.06,151.87,224151.84,14777,4.76,11.09
VL_ITEM_PAGO_FORNECEDOR,295563.0,33.75,362.75,0.0,0.00,0.00,0.00,22000.00,14777,4.76,34.19
VL_TOTAL_REMUNERACAO,290637.0,842.46,4409.86,0.0,118.09,172.03,326.24,157496.59,19703,6.35,12.71
N_LINHAS_REM,290637.0,1.02,0.14,1.0,1.00,1.00,1.00,2.00,19703,6.35,7.04
VL_ITEM_EVENTO_INFORMADO_CAPPED,295563.0,716.45,3741.43,0.0,65.66,85.06,151.87,35068.52,14777,4.76,7.49
VL_ITEM_PAGO_FORNECEDOR_CAPPED,295563.0,32.36,305.04,0.0,0.00,0.00,0.00,11541.49,14777,4.76,28.60
VL_TOTAL_REMUNERACAO_CAPPED,290637.0,809.25,3830.83,0.0,118.09,172.03,326.24,53634.23,19703,6.35,9.81
QT_ITEM_EVENTO_INFORMADO_CAPPED,295563.0,8.78,39.26,1.0,1.00,1.00,1.00,331.61,14777,4.76,6.44
VL_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,295563.0,0.45,0.11,0.0,0.40,0.43,0.48,1.00,14777,4.76,2.74


**Findings:** 

- **Missing values:** the four raw item-level columns (`QT_ITEM_EVENTO_INFORMADO`, 
  `VL_ITEM_EVENTO_INFORMADO`, `VL_ITEM_PAGO_FORNECEDOR`, and their `_CAPPED` 
  variants) share an identical 4.76% missing rate. This is fully explained by 
  bundled procedures (`IND_PACOTE=1`), which have no individual quantity/value 
  reported. `VL_TOTAL_REMUNERACAO` and `N_LINHAS_REM` show a slightly higher 
  6.35% missing rate, reflecting events with no matching row in the REM file. 
  These missing-value rates are the ones behind the missing-value treatment
  decisions taken in notebook 04.

- **Skewness:** raw monetary columns are heavily right-skewed (e.g. 
  `VL_ITEM_PAGO_FORNECEDOR`, skew=34.19; `VL_ITEM_EVENTO_INFORMADO`, skew=11.09), 
  driven by a small number of very high-value outliers. The capped and 
  log-normalized variants substantially reduce this skew (e.g. down to 2.21 
  for the payment column), confirming the transformations applied during 
  feature engineering were effective.

- **Central tendency vs. dispersion gap:** `VL_ITEM_PAGO_FORNECEDOR` has a 
  median of 0 despite a mean of R$33.75 — more than half of items had no 
  direct supplier payment recorded, while a small share of high-value payments 
  pull the mean upward. This asymmetry should be considered before using 
  mean-based aggregations in later hypothesis testing.

#### Categorical columns

Frequency distributions (counts and percentages per category) were computed 
for all 103 categorical columns, including missing-value rates. Columns with 
15 or fewer unique categories (97 columns) are documented with their full 
distribution; the remaining 6 high-cardinality columns are summarized as the 
top 10 categories plus an aggregated "OUTROS" bucket, to keep the notebook 
readable.

In [5]:
LOW_CARDINALITY_THRESHOLD = 15

low_card_cols = df_classification[df_classification["final_classification"].str.startswith("categorical") & (df_classification["n_unique"] <= LOW_CARDINALITY_THRESHOLD)]["column"].tolist()
high_card_cols = df_classification[df_classification["final_classification"].str.startswith("categorical") & (df_classification["n_unique"] > LOW_CARDINALITY_THRESHOLD)]["column"].tolist()

print(f"Low-cardinality columns (full distribution): {len(low_card_cols)}")
print(f"High-cardinality columns (top-N summary): {len(high_card_cols)}")

Low-cardinality columns (full distribution): 97
High-cardinality columns (top-N summary): 6


In [6]:
def get_full_distribution(df, col):
    counts = df[col].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(2)
    dist = pd.DataFrame({"count": counts, "pct": pct})
    dist.index.name = col
    
    missing_pct = round(df[col].isna().mean() * 100, 2)
    print(f"{col}: {df[col].nunique()} categories | missing: {missing_pct}%")
    
    return dist

low_card_distributions = {}
for col in low_card_cols:
    low_card_distributions[col] = get_full_distribution(df_norte, col)

print(f"\nDistributions generated for {len(low_card_distributions)} columns.")

UF_PRESTADOR: 7 categories | missing: 0.0%
CD_TABELA_REFERENCIA: 3 categories | missing: 0.0%
FAIXA_ETARIA: 7 categories | missing: 0.0%
PORTE: 4 categories | missing: 0.0%
CD_MODELO_REMUNERACAO: 4 categories | missing: 6.35%


treatment_line: 10 categories | missing: 0.0%


subtype: 8 categories | missing: 0.0%
IND_LINE_ADJUVANT: 2 categories | missing: 0.0%
IND_LINE_ADVANCED: 2 categories | missing: 0.0%
IND_LINE_NEOADJUVANT: 2 categories | missing: 0.0%
IND_LINE_STAGING: 2 categories | missing: 0.0%
IND_SUBTYPE_BRCA_MUTATED: 2 categories | missing: 0.0%
IND_SUBTYPE_HER2_UNDETERMINED: 2 categories | missing: 0.0%
IND_SUBTYPE_HER2POS: 2 categories | missing: 0.0%
IND_SUBTYPE_HER2POS_HER2_LOW: 2 categories | missing: 0.0%
IND_SUBTYPE_HRPOS_HER2_: 2 categories | missing: 0.0%
IND_SUBTYPE_NTRKPOS: 2 categories | missing: 0.0%


IND_SUBTYPE_GENERAL: 2 categories | missing: 0.0%
IND_SUBTYPE_TRIPLE_NEGATIVE: 2 categories | missing: 0.0%


IND_REIMBURSEMENT_MODEL_01: 2 categories | missing: 0.0%
IND_REIMBURSEMENT_MODEL_02: 2 categories | missing: 0.0%
IND_REIMBURSEMENT_MODEL_04: 2 categories | missing: 0.0%
IND_REIMBURSEMENT_REPORTED: 2 categories | missing: 0.0%
SEXO_Feminino: 2 categories | missing: 0.0%
SEXO_Masculino: 2 categories | missing: 0.0%
IND_PACOTE_0: 2 categories | missing: 0.0%
IND_PACOTE_1: 2 categories | missing: 0.0%
CD_CARATER_ATENDIMENTO_1: 2 categories | missing: 0.0%
CD_CARATER_ATENDIMENTO_2: 2 categories | missing: 0.0%


...

NM_MODALIDADE_Seguradora Especializada Em Saúde: 2 categories | missing: 0.0%
UF_AC: 2 categories | missing: 0.0%
UF_AM: 2 categories | missing: 0.0%
UF_AP: 2 categories | missing: 0.0%
UF_PA: 2 categories | missing: 0.0%
UF_RO: 2 categories | missing: 0.0%
UF_RR: 2 categories | missing: 0.0%
UF_TO: 2 categories | missing: 0.0%
TIPO_CONSULTA_1: 2 categories | missing: 0.0%


TIPO_CONSULTA_2: 2 categories | missing: 0.0%
TIPO_CONSULTA_3: 2 categories | missing: 0.0%
TIPO_CONSULTA_4: 2 categories | missing: 0.0%
TIPO_CONSULTA_NOT_APPLICABLE_NOT_A_CONSULTATION: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_001: 2 categories | missing: 0.0%


UNIDADE_MEDIDA_002: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_004: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_005: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_006: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_008: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_013: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_014: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_015: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_021: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_022: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_023: 2 categories | missing: 0.0%


UNIDADE_MEDIDA_024: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_031: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_032: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_036: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_037: 2 categories | missing: 0.0%
UNIDADE_MEDIDA_NOT_APPLICABLE_PROCEDURE: 2 categories | missing: 0.0%


category_drug: 2 categories | missing: 0.0%
category_procedure: 2 categories | missing: 0.0%
category_procedure_group: 2 categories | missing: 0.0%
FAIXA_ETARIA_ORDINAL: 7 categories | missing: 0.0%
PORTE_ORDINAL: 4 categories | missing: 0.0%
IND_SIZE_NO_BENEFICIARIES: 2 categories | missing: 0.0%
IND_HIGH_SPECIFICITY_PROXY: 2 categories | missing: 0.0%

Distributions generated for 97 columns.


In [7]:
low_card_distributions["UF_PRESTADOR"]

,count,pct
UF_PRESTADOR,,
PA,155153,49.99
AM,78613,25.33
RO,28687,9.24
TO,22527,7.26
AP,14351,4.62
AC,5691,1.83
RR,5318,1.71


In [8]:
low_card_distributions["FAIXA_ETARIA"]

,count,pct
FAIXA_ETARIA,,
40 a 49 anos,104085,33.54
50 a 59 anos,68318,22.01
30 a 39 anos,47905,15.44
60 a 69 anos,42516,13.70
20 a 29 anos,22022,7.10
70 a 79 anos,19945,6.43
80 anos ou mais,5549,1.79


**Findings:** `UF_PRESTADOR` shows strong geographic concentration, with PA 
(49.99%) and AM (25.33%) accounting for ~75% of all events combined — the 
remaining 5 states share the rest. 

`FAIXA_ETARIA` peaks in the 40-59 age range (55.6% combined), consistent with 
known breast cancer diagnosis age patterns. This dataset reflects an age filter 
applied upstream, restricting the population to 7 age brackets (20 to 80+ years) 
rather than the full range originally present in the raw data.

In [9]:
TOP_N = 10

def get_topn_summary(df, col, top_n=TOP_N):
    counts = df[col].value_counts(dropna=False)
    total = len(df)
    
    top = counts.head(top_n)
    top_pct = (top / total * 100).round(2)
    
    outros_count = counts.iloc[top_n:].sum()
    outros_pct = round(outros_count / total * 100, 2)
    
    summary = pd.DataFrame({"count": top, "pct": top_pct})
    summary.loc["OUTROS"] = [outros_count, outros_pct]
    
    n_unique_total = df[col].nunique()
    cobertura_topn = round(top.sum() / total * 100, 2)
    missing_pct = round(df[col].isna().mean() * 100, 2)
    
    print(f"{col}: {n_unique_total} unique values | top {top_n} cover {cobertura_topn}% | missing: {missing_pct}%")
    
    return summary

In [10]:
high_card_summaries = {}
high_card_summaries["CD_PROCEDIMENTO"] = get_topn_summary(df_norte, "CD_PROCEDIMENTO")
high_card_summaries["CD_PROCEDIMENTO"]

CD_PROCEDIMENTO: 155 unique values | top 10 cover 94.48% | missing: 0.0%


,count,pct
CD_PROCEDIMENTO,,
40901114,145816.0,46.99
40808041,76473.0,24.64
40808033,47868.0,15.42
022,5537.0,1.78
41101480,4415.0,1.42
40601170,3959.0,1.28
90069137,3010.0,0.97
90066235,2573.0,0.83
021,2081.0,0.67


**Findings:** despite 155 unique procedure codes, the top 10 account for 
94.48% of events, with the top 3 alone (`40901114`, `40808041`, `40808033`) 
covering ~87%. This extreme concentration indicates that a small set of 
dominant procedures drives most of the volume, and should inform which 
procedures merit deeper investigation ahead of hypothesis generation. The 
remaining 145 procedure codes appear rarely and are aggregated into "OUTROS".

#### Temporal columns

Descriptive statistics for the 2 temporal columns (`DT_REALIZACAO`, `MES`) 
include date range, number of unique periods, and missing rate.

In [11]:
temporal_stats_rows = []
for col in temporal_cols:
    temporal_stats_rows.append({
        "column": col,
        "min_date": df_norte[col].min(),
        "max_date": df_norte[col].max(),
        "n_unique": df_norte[col].nunique(),
        "missing": df_norte[col].isna().sum(),
        "missing_pct": round(df_norte[col].isna().mean() * 100, 2),
    })

temporal_stats = pd.DataFrame(temporal_stats_rows)
temporal_stats

,column,min_date,max_date,n_unique,missing,missing_pct
0,DT_REALIZACAO,2025-01-01,2025-12-01,12,0,0.0
1,MES,2025-01-01,2025-12-01,12,0,0.0


In [12]:
df_norte["DT_REALIZACAO"].dt.to_period("M").value_counts().sort_index()

DT_REALIZACAO
2025-01    26499
2025-02    26123
2025-03    25747
2025-04    26754
2025-05    28758
2025-06    26587
2025-07    28863
2025-08    27722
2025-09    26758
2025-10    27708
2025-11    21357
2025-12    17464
Freq: M, Name: count, dtype: int64

**Findings:** `DT_REALIZACAO` and `MES` both span the full 12 months of 2025 
(January to December) with zero missing values, and hold identical values 
across all rows: consistent with `MES` being derived directly from 
`DT_REALIZACAO` during ingestion, making the two effectively redundant.

Monthly volume is relatively stable from January through October 
(~26,000-29,000 events/month), but drops noticeably in November (21,357) and 
December (17,464). This is most likely a reporting-lag artifact, ANS data 
for the most recent months of a given year is typically still being submitted 
by operators at the time of extraction, rather than a genuine decline in 
care volume, and should be considered when interpreting any trend analysis 
that includes Q4 2025.

In [13]:
col = "ID_EVENTO_ATENCAO_SAUDE"
key_stats = pd.DataFrame([{
    "column": col,
    "n_unique": df_norte[col].nunique(),
    "n_rows": len(df_norte),
    "is_fully_unique": df_norte[col].nunique() == len(df_norte),
    "missing": df_norte[col].isna().sum(),
    "missing_pct": round(df_norte[col].isna().mean() * 100, 2),
    "note": "Unique event identifier -- not applicable for descriptive statistics",
}])
key_stats

,column,n_unique,n_rows,is_fully_unique,missing,missing_pct,note
0,ID_EVENTO_ATENCAO_SAUDE,291864,310340,False,0,0.0,Unique event identifier -- not applicable for ...


In [14]:

dup_counts = df_norte["ID_EVENTO_ATENCAO_SAUDE"].value_counts()
dup_counts[dup_counts > 1].describe()

count    15062.000000
mean         2.226663
std          0.973324
min          2.000000
25%          2.000000
50%          2.000000
75%          2.000000
max          7.000000
Name: count, dtype: float64

In [15]:
dup_counts = df_norte["ID_EVENTO_ATENCAO_SAUDE"].value_counts()
exemplo_id = dup_counts[dup_counts > 1].index[0]
df_norte[df_norte["ID_EVENTO_ATENCAO_SAUDE"] == exemplo_id]

,ID_EVENTO_ATENCAO_SAUDE,UF_PRESTADOR,DT_REALIZACAO,CD_PROCEDIMENTO,CD_TABELA_REFERENCIA,QT_ITEM_EVENTO_INFORMADO,VL_ITEM_EVENTO_INFORMADO,VL_ITEM_PAGO_FORNECEDOR,ID_PLANO,FAIXA_ETARIA,...,category_procedure,category_procedure_group,FAIXA_ETARIA_ORDINAL,PORTE_ORDINAL,IND_SIZE_NO_BENEFICIARIES,VL_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,VL_ITEM_PAGO_FORNECEDOR_CAPPED_LOG_NORM,VL_TOTAL_REMUNERACAO_CAPPED_LOG_NORM,QT_ITEM_EVENTO_INFORMADO_CAPPED_LOG_NORM,IND_HIGH_SPECIFICITY_PROXY
5766,7477773801,AM,2025-01-01,40901114,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5767,7477773801,AM,2025-01-01,40601170,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5768,7477773801,AM,2025-01-01,40601188,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5769,7477773801,AM,2025-01-01,40602010,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5770,7477773801,AM,2025-01-01,40808173,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5807,7477773801,AM,2025-01-01,40808033,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0
5808,7477773801,AM,2025-01-01,40808041,22,NaN,NaN,NaN,214968,50 a 59 anos,...,1,0,8,2,0,NaN,NaN,0.522533,NaN,0


**Findings:** `ID_EVENTO_ATENCAO_SAUDE` has zero missing values, but is 
**not** fully unique at the row level (291,864 unique values across 310,340 
rows). This is expected, not a data quality issue: the DET source operates at 
item/procedure level, and bundled events (`IND_PACOTE=1`) generate one row 
per procedure code within the same event -- confirmed by inspecting a 
duplicated ID, where each row carries a different `CD_PROCEDIMENTO` while 
`QT_ITEM_EVENTO_INFORMADO`/`VL_ITEM_EVENTO_INFORMADO` are null, consistent 
with the bundling pattern already documented during data preparation.

This means `ID_EVENTO_ATENCAO_SAUDE` should be treated as an **event-level** 
key, not a **row-level** key: any per-event aggregation (e.g. counting 
distinct patients/events) must first deduplicate or group by this column, 
while row-level analyses (e.g. per-procedure statistics) can use the data as-is.

## Reproducibility note

The notebook requires Python with Pandas and PyArrow (the Parquet engine). It
installs nothing and makes no network request: it reads only the treated Parquet
produced by notebook 04, so the pinned environment of `requirements.txt` is
enough to reproduce every number below.

## References

1. Agência Nacional de Saúde Suplementar (ANS). Dados abertos: TISS Ambulatorial [Internet]. Brasília: ANS; 2026 [cited 2026 Sep 11]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL/
2. Agência Nacional de Saúde Suplementar (ANS). Padrão TISS: dicionário de variáveis. Brasília: ANS; 2024.
